# Week 13: Neural networks from scratch (activations, autograd, backprop)

**This week answers your question about activation functions**, and turns backpropagation from a formula you memorised into code you wrote.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Why non-linearity? Activation functions and their derivatives | 2 h |
| Day 2 | A tiny autograd engine (micrograd-style): backprop as the chain rule on a graph | 3 h |
| Day 3 | A 2-layer MLP in NumPy: derive and implement the backward pass | 3 h |
| Day 4 | Train it: mini-batches, initialisation, decision boundaries | 1.5 h |
| Review | Spaced recall and a derivation from a blank page | 0.5 h |

**Watch (essential):** Andrej Karpathy, *The spelled-out intro to neural networks and backpropagation: building micrograd* (about 2.5 h; spread it over Days 1–2 and code along). **Read:** Nielsen, *Neural Networks and Deep Learning*, chapter 2.

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can explain why 10 stacked linear layers equal one linear layer.
- [ ] You can derive $\partial L/\partial W_1$ for a 2-layer MLP with ReLU and softmax cross-entropy on paper.
- [ ] Your NumPy MLP passes the gradient check and reaches at least 95% validation accuracy on two moons.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
# Day 1: Why do we need non-linear activations? (2 h)

A linear layer computes $h = Wx + b$. Stack two of them: $W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2)$, which is **still a single linear map**. Without non-linearities, a 100-layer network can only draw straight decision boundaries, just like logistic regression.

An **activation function** bends the space between the layers, so that later layers can carve out curved regions. The usual choices:

| Activation | Formula | Derivative | Notes |
|---|---|---|---|
| sigmoid | $1/(1 + e^{-z})$ | $\sigma(1 - \sigma)$ | outputs a probability; **saturates**, since the gradient is at most 0.25 and vanishes for large |z| |
| tanh | $\tanh z$ | $1 - \tanh^2 z$ | zero-centred, but still saturates |
| **ReLU** | $\max(0, z)$ | $1$ if $z > 0$, else $0$ | the default in hidden layers: cheap, no saturation for z > 0; units can "die" |
| GELU | $z\,\Phi(z)$ | — | a smooth ReLU, used in transformers |
| softmax | $e^{z_i} / \sum_j e^{z_j}$ | — | **output layer** for multi-class: turns scores into probabilities (Week 7: categorical MLE) |

**The rule of thumb:** ReLU (or GELU) in hidden layers; at the output, *whatever matches the likelihood of your labels* (sigmoid for binary, softmax for multi-class, none for regression).

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
X, y = make_moons(600, noise=0.2, random_state=0)
lin = LogisticRegression().fit(X, y)
print("logistic regression accuracy on moons:", round(lin.score(X, y), 3), "← a straight line can't separate them")
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for name, f, df in [("sigmoid", lambda z: 1 / (1 + np.exp(-z)), lambda z: (1 / (1 + np.exp(-z))) * (1 - 1 / (1 + np.exp(-z)))),
                    ("tanh", np.tanh, lambda z: 1 - np.tanh(z) ** 2),
                    ("relu", lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float))]:
    ax[0].plot(z, f(z), label=name); ax[1].plot(z, df(z), label=name)
ax[0].set_title("activation"); ax[1].set_title("derivative"); ax[0].legend(); plt.show()

✍️ *Look at the derivative plot. In a 20-layer sigmoid network, the backward pass multiplies ~20 such derivatives together. What happens to the gradient reaching the first layer? That's the vanishing-gradient problem, and why ReLU took over.* (double-click to edit)



### Exercise 13.1: Activations and their derivatives
Implement them vectorised. `softmax` must be **numerically stable**: subtract the row maximum before exponentiating (it doesn't change the result; why?).

In [ ]:
def relu(z):
    raise NotImplementedError

def relu_grad(z):
    raise NotImplementedError

def tanh_grad(z):
    raise NotImplementedError

def softmax(Z):
    """Row-wise softmax for a 2-D array of scores."""
    raise NotImplementedError

In [ ]:
z = np.array([-2.0, -0.1, 0.5, 3.0])
assert np.array_equal(relu(z), [0, 0, 0.5, 3.0]) and np.array_equal(relu_grad(z), [0, 0, 1, 1])
h = 1e-6
assert np.allclose(tanh_grad(z), (np.tanh(z + h) - np.tanh(z - h)) / (2 * h), atol=1e-8)
S = softmax(np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]]))
assert np.allclose(S.sum(axis=1), 1) and np.allclose(S[1], 1 / 3) and not np.isnan(S).any()
print("✅ 13.1 passed")

---
# Day 2: A tiny autograd engine (3 h)

Every number in a computation becomes a **node** that remembers its inputs and how to send gradients back to them. Calling `loss.backward()`:
1. sorts the graph **topologically** (every node comes after its inputs),
2. sets `loss.grad = 1`,
3. walks the nodes in **reverse** order, and each one applies the **chain rule** locally: `input.grad += local_derivative * self.grad`.

The `+=` matters: if a value is used twice, its gradients from both uses **add up** (the multivariable chain rule).

This is *exactly* what PyTorch does, only with tensors instead of scalars. `__add__` is done for you. Complete the rest.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad          # d(a+b)/da = 1
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        raise NotImplementedError             # d(a*b)/da = b

    def tanh(self):
        raise NotImplementedError             # d tanh(x)/dx = 1 - tanh(x)^2

    def exp(self):
        raise NotImplementedError

    def backward(self):
        raise NotImplementedError             # topological sort, then call _backward in reverse

    # conveniences that reuse the operations above
    __radd__ = __add__
    def __rmul__(self, other): return self * other
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)

In [ ]:
# a single neuron: tanh(w1*x1 + w2*x2 + b)
x1, x2, w1, w2, b = Value(2.0), Value(0.0), Value(-3.0), Value(1.0), Value(6.8813735870195432)
o = (x1 * w1 + x2 * w2 + b).tanh()
o.backward()
assert abs(o.data - 0.7071) < 1e-4
assert abs(w1.grad - 1.0) < 1e-4 and abs(x1.grad - (-1.5)) < 1e-4 and abs(w2.grad) < 1e-9

a = Value(3.0); c = a * a + a.exp()           # a is used three times: gradients must accumulate
c.backward()
assert abs(a.grad - (2 * 3.0 + math.exp(3.0))) < 1e-9
print("✅ 13.2 passed:", w1, x1)

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`__mul__`: `out = Value(self.data * other.data, (self, other), '*')`. In `_backward`: `self.grad += other.data * out.grad` and `other.grad += self.data * out.grad`.

</details>

<details><summary>Hint 2</summary>

`tanh`: compute `t = math.tanh(self.data)` once; the backward adds `(1 - t ** 2) * out.grad`. `exp` is similar, with derivative `out.data`.

</details>

<details><summary>Hint 3</summary>

`backward`: build `topo = []` with a recursive `build(v)` that visits `v._prev` first (using a `visited` set), then appends v. Set `self.grad = 1.0`, then `for v in reversed(topo): v._backward()`.

</details>


---
# Day 3: A 2-layer MLP in NumPy, in matrix form (3 h)

Scalars are too slow for real networks. Here's the same chain rule with matrices. For a batch $X$ ($n \times d$) and one-hot labels $Y$ ($n \times C$):

**Forward:**
$$Z_1 = XW_1 + b_1, \quad H = \text{ReLU}(Z_1), \quad Z_2 = HW_2 + b_2, \quad P = \text{softmax}(Z_2), \quad L = -\tfrac{1}{n}\textstyle\sum Y \odot \log P$$

**Backward** (derive each line on paper; the first one is the famous simplification of softmax + cross-entropy):
$$\frac{\partial L}{\partial Z_2} = \frac{P - Y}{n}, \quad \frac{\partial L}{\partial W_2} = H^\top \frac{\partial L}{\partial Z_2}, \quad \frac{\partial L}{\partial b_2} = \textstyle\sum_{\text{rows}} \frac{\partial L}{\partial Z_2}$$
$$\frac{\partial L}{\partial H} = \frac{\partial L}{\partial Z_2} W_2^\top, \quad \frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial H} \odot \text{ReLU}'(Z_1), \quad \frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial Z_1}, \quad \frac{\partial L}{\partial b_1} = \textstyle\sum_{\text{rows}} \frac{\partial L}{\partial Z_1}$$

**Shape check:** each gradient has the **same shape as the thing it's the gradient of**. When you're lost, write the shapes down and the transposes become obvious.

### Exercise 13.3: `forward` and `backward`
`params` is a dict with keys `W1, b1, W2, b2`. `forward` returns `(loss, cache)`, where the cache holds whatever `backward` needs. `backward(params, cache)` returns a dict of gradients with the same keys. Use your `relu`, `relu_grad` and `softmax`.

In [ ]:
def init_params(d_in, d_hidden, d_out, seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, np.sqrt(2 / d_in), (d_in, d_hidden)), "b1": np.zeros(d_hidden),      # He initialisation
            "W2": r.normal(0, np.sqrt(2 / d_hidden), (d_hidden, d_out)), "b2": np.zeros(d_out)}

def forward(params, X, Y):
    raise NotImplementedError

def backward(params, cache):
    raise NotImplementedError

In [ ]:
Xs = rng.normal(size=(8, 3)); Ys = np.eye(4)[rng.integers(0, 4, 8)]
p = init_params(3, 5, 4)
loss, cache = forward(p, Xs, Ys)
grads = backward(p, cache)
for k in p:                                    # gradient check, parameter by parameter
    num = np.zeros_like(p[k]); it = np.nditer(p[k], flags=["multi_index"])
    for _ in it:
        i = it.multi_index; old = p[k][i]
        p[k][i] = old + 1e-6; lp, _ = forward(p, Xs, Ys)
        p[k][i] = old - 1e-6; lm, _ = forward(p, Xs, Ys)
        p[k][i] = old; num[i] = (lp - lm) / 2e-6
    assert grads[k].shape == p[k].shape, k
    assert np.allclose(grads[k], num, atol=1e-6), f"gradient of {k} is wrong"
print("✅ 13.3 passed: all four gradients match the numerical check")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Cache `X, Z1, H, P, Y`. The loss is `-np.mean(np.sum(Y * np.log(P + 1e-12), axis=1))`.

</details>

<details><summary>Hint 2</summary>

Backward: `dZ2 = (P - Y) / n`, `dW2 = H.T @ dZ2`, `db2 = dZ2.sum(axis=0)`, `dH = dZ2 @ W2.T`, `dZ1 = dH * relu_grad(Z1)`, and so on.

</details>


---
# Day 4: Train it (1.5 h)

### Exercise 13.4: `train`
Mini-batch SGD: each epoch, shuffle the training indices, loop over batches of `batch_size`, run forward/backward, and update every parameter with `p[k] -= lr * g[k]`. Return `(params, history)`, where history is a list of the training loss per epoch.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = make_moons(1000, noise=0.2, random_state=1)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=0)
Y_tr = np.eye(2)[y_tr]

def train(X, Y, d_hidden=32, lr=0.5, epochs=200, batch_size=32, seed=0):
    raise NotImplementedError

def predict(params, X):
    H = relu(X @ params["W1"] + params["b1"])
    return (H @ params["W2"] + params["b2"]).argmax(axis=1)

In [ ]:
params, hist = train(X_tr, Y_tr)
acc = (predict(params, X_va) == y_va).mean()
print("validation accuracy:", acc)
assert acc >= 0.95 and hist[-1] < hist[0]
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
zz = predict(params, np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].contourf(xx, yy, zz, alpha=0.3); ax[0].scatter(X_va[:, 0], X_va[:, 1], c=y_va, s=8); ax[0].set_title("decision boundary")
ax[1].plot(hist); ax[1].set_title("training loss"); ax[1].set_xlabel("epoch"); plt.show()
print("✅ 13.4 passed")

✍️ *Experiments (write down what happens and why): (1) initialise all weights to zero; (2) replace ReLU with nothing (identity); (3) lr = 5; (4) d_hidden = 2.* (double-click to edit)



---
# Review (30 min)
**Blank page (on paper):** derive the backward pass of the 2-layer MLP, including why $\partial L/\partial Z_2 = (P - Y)/n$ for softmax + cross-entropy.

1. *(Week 6)* What does the chain rule say?
2. *(Week 7)* Cross-entropy is the negative log-likelihood of which distribution?
3. *(Week 12)* What's RRF, and why doesn't it need score normalisation?
4. Why do gradients accumulate (`+=`) in autograd?
5. Why does zero initialisation fail? (Symmetry.)
6. Sigmoid versus ReLU in hidden layers: which one, and why?

✍️ *Your answer:* (double-click to edit)

